# Homework 1 — Submission: LightGBM baseline with the building

**lgbm_building** alone, to score the baseline on the leaderboard next to the blend in
[submission.ipynb](submission.ipynb). Same model as the blend's LightGBM half: LightGBM on the 33 features plus
`building_id` as a categorical, untuned (500 trees, learning rate 0.1, 63 leaves). On the 4 seasonal folds it scored
1.217, against 1.187 for mlp_4out_emb and 1.178 for the blend ([ensemble.ipynb](ensemble.ipynb)).

- **Training rows:** the same 2M rows as [submission.ipynb](submission.ipynb): sampled from all of 2016 with anomalies
  dropped, same seed.
- **No torch here,** so LightGBM can use every core (next to torch it crashes unless limited to one thread).

Output: `data/processed/submission_lgbm_building.csv`, with site 0 electricity converted back to kBTU, the unit Kaggle
scores.

In [1]:
import sys

import lightgbm as lgb
import mlflow
import numpy as np
import pandas as pd

sys.path.insert(0, "..")  # the ashrae package lives in the homework folder
from ashrae.config import DATA, KBTU_TO_KWH, MLFLOW_URI, PROCESSED
from ashrae.data import read_processed
from ashrae.features import build_features

N_TRAIN = 2_000_000
LGBM_PARAMS = {"n_estimators": 500, "learning_rate": 0.1, "num_leaves": 63}
mlflow.set_tracking_uri(MLFLOW_URI)
_ = mlflow.set_experiment("ashrae-submission")

2026/09/27 22:59:35 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/lupaskodmitro/FuckIT/Machine Learning/Deep Learning Course/.venv/lib/python3.14/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


## Training rows

Sampled from all of 2016 without anomalies ([EDA §9](../eda/EDA.ipynb#broken)), as in
[submission.ipynb](submission.ipynb). The `year_built` median comes from these rows and is reused for the test set.

In [2]:
df = read_processed()
n_buildings = int(df.building_id.max()) + 1
BUILDING = pd.CategoricalDtype(range(n_buildings))
train_df = df[~df.anomaly].sample(N_TRAIN, random_state=0)
del df
year_median = train_df.year_built.median()
X_train = build_features(train_df, year_median)
y_train = np.log1p(train_df.meter_reading.to_numpy(dtype="float32"))
print(f"{len(train_df):,} training rows, {X_train.shape[1]} features + building_id")


def with_building(X, rows):
    return X.assign(building_id=rows.building_id.astype(int).astype(BUILDING).to_numpy())

2,000,000 training rows, 33 features + building_id


## Model

In [3]:
lgbm = lgb.LGBMRegressor(**LGBM_PARAMS, random_state=0, verbose=-1).fit(with_building(X_train, train_df), y_train)
train_pred = np.clip(lgbm.predict(with_building(X_train, train_df)), 0, None)
train_rmsle = float(np.sqrt(np.mean((train_pred - y_train) ** 2)))
with mlflow.start_run(run_name=f"lgbm_building | {N_TRAIN:,} rows | submission",
                      tags={"model": "lgbm_building", "notebook": "submission_baseline"}):
    mlflow.log_params({**LGBM_PARAMS, "n_train": N_TRAIN})
    mlflow.log_metric("train_rmsle", train_rmsle)
print(f"lgbm_building: training RMSLE {train_rmsle:.3f}")

lgbm_building: training RMSLE 0.833


## Test predictions

The 41.7M test rows in chunks of 1M. `log1p` predictions are clipped at 0 and turned back into readings with `expm1`.

In [4]:
test_df = read_processed("test_merged.parquet")


def predict_chunk(chunk):
    X = build_features(chunk, year_median).reindex(columns=X_train.columns, fill_value=0)
    return np.clip(lgbm.predict(with_building(X, chunk)), 0, None)


lgbm_log = np.concatenate([predict_chunk(test_df.iloc[i : i + 1_000_000]) for i in range(0, len(test_df), 1_000_000)])
print(f"mean log1p prediction: {lgbm_log.mean():.3f}")

mean log1p prediction: 4.234


In [5]:
reading = np.expm1(lgbm_log)
reading[((test_df.site_id == 0) & (test_df.meter == 0)).to_numpy()] /= KBTU_TO_KWH  # back to kBTU

submission = pd.DataFrame({"row_id": test_df.row_id.to_numpy(), "meter_reading": reading}).sort_values("row_id")
sample_ids = pd.read_csv(DATA / "sample_submission.csv", usecols=["row_id"]).row_id
assert np.array_equal(submission.row_id, sample_ids) and np.isfinite(submission.meter_reading).all()
assert (submission.meter_reading >= 0).all()
submission.to_csv(PROCESSED / "submission_lgbm_building.csv", index=False, float_format="%.4f")
print(f"saved {PROCESSED / 'submission_lgbm_building.csv'}: {len(submission):,} rows")
submission.meter_reading.describe().round(2).to_frame()

saved /Users/lupaskodmitro/FuckIT/Machine Learning/Deep Learning Course/Homeworks/Homework 1/data/processed/submission_lgbm_building.csv: 41,697,600 rows


,meter_reading
count,41697600.00
mean,376.83
std,2710.44
min,0.00
25%,22.76
50%,76.16
75%,229.89
max,1734915.79


## Next

Upload `submission_lgbm_building.csv` to Kaggle and compare with the blend (private 1.302, public 1.086). If the
leaderboard ranks them like the folds did (1.217 vs 1.178), the fold validation at least ranks models correctly.